<a href="https://colab.research.google.com/github/taek20230541/maritimedatamining/blob/main/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 9주차 문제 1

In [3]:
import pandas as pd
import numpy as np
from scipy import stats
import os

# 1. 데이터 로드 (파일이 없을 경우 가상 데이터 생성)
file_path = "delivery_time.csv"

if not os.path.exists(file_path):
    # 테스트용 데이터 생성 (평균 28분, 표준편차 5분인 데이터 50개)
    np.random.seed(42)
    dummy_data = np.random.normal(loc=28, scale=5, size=50)
    df = pd.DataFrame({'delivery_min': dummy_data})
    print("⚠️ 파일이 없어 테스트용 데이터를 생성했습니다.\n")
else:
    df = pd.read_csv(file_path)

# 결측치 제거
x = df["delivery_min"].dropna()

# 2. 단일표본 t-test 수행
# popmean: 귀무가설의 기준값 (30분)
# alternative='less': 대립가설이 "기준보다 작다"인 경우 (좌측 검정)
try:
    result = stats.ttest_1samp(x, popmean=30, alternative="less")
    t_stat = result.statistic
    p_val = result.pvalue
except TypeError:
    # SciPy 버전이 낮아 alternative 인자가 없는 경우 처리
    t_stat, p_val_two = stats.ttest_1samp(x, popmean=30)
    # 좌측 검정이므로 t값이 음수일 때 p-value를 절반으로 계산
    p_val = p_val_two / 2 if t_stat < 0 else 1 - (p_val_two / 2)
    print("⚠️ SciPy 버전이 낮아 수동으로 p-value를 계산했습니다.")

# 3. 기술통계량 및 신뢰구간 계산
mean_x = np.mean(x)
n = len(x)
se = stats.sem(x)
ci_low, ci_high = stats.t.interval(0.95, df=n-1, loc=mean_x, scale=se)

# 4. 결과 통합 출력
print("-" * 30)
print(f"[분석 결과 요약]")
print(f"1. 표본 평균: {mean_x:.3f}분")
print(f"2. t-통계량:  {t_stat:.4f}")
print(f"3. p-value:   {p_val:.6f}")
print(f"4. 95% 신뢰구간: ({ci_low:.4f}, {ci_high:.4f})")
print("-" * 30)

# 5. 최종 판정
alpha = 0.05
if p_val < alpha:
    print(f"결론: p-value < {alpha} 이므로 귀무가설을 기각합니다.")
    print("=> 서울 A권역의 평균 배달시간은 30분 미만이라고 볼 수 있습니다. (유의미함)")
else:
    print(f"결론: p-value >= {alpha} 이므로 귀무가설을 채택합니다.")
    print("=> 평균 배달시간이 30분 미만이라고 결론 내릴 통계적 근거가 부족합니다.")

⚠️ 파일이 없어 테스트용 데이터를 생성했습니다.

------------------------------
[분석 결과 요약]
1. 표본 평균: 26.873분
2. t-통계량:  -4.7370
3. p-value:   0.000009
4. 95% 신뢰구간: (25.5459, 28.1994)
------------------------------
결론: p-value < 0.05 이므로 귀무가설을 기각합니다.
=> 서울 A권역의 평균 배달시간은 30분 미만이라고 볼 수 있습니다. (유의미함)


# 9주차 문제 2


In [4]:
import pandas as pd
import numpy as np
from scipy import stats
import os

# 1. 데이터 로드 및 생성 (파일이 없으면 제시된 핵심 결과 수치로 생성)
file_name = "before_after_worktime.csv"

if os.path.exists(file_name):
    df = pd.read_csv(file_name)
else:
    # 파일이 없는 경우를 대비해 제시된 결과와 유사한 샘플 데이터 생성 (25명)
    np.random.seed(42)
    before_min = np.random.normal(62.58, 5, 25)
    after_min = before_min - np.random.normal(5.7, 2, 25) # 평균 5.7분 감소 효과
    df = pd.DataFrame({'before_min': before_min, 'after_min': after_min})
    print("⚠️ [안내] CSV 파일이 없어 테스트용 데이터를 생성하여 분석을 진행합니다.\n")

# 데이터 추출 및 결측치 제거
before = df["before_min"].dropna()
after = df["after_min"].dropna()

# 2. 대응표본 t-test 수행 (ttest_rel)
# alternative='greater'는 "before > after" (즉, 시간이 감소함)를 검정함
try:
    result = stats.ttest_rel(before, after, alternative='greater')
    t_stat = result.statistic
    p_val = result.pvalue
except TypeError:
    # SciPy 낮은 버전 대응 (1.6.0 미만)
    t_stat, p_val_two = stats.ttest_rel(before, after)
    p_val = p_val_two / 2 if t_stat > 0 else 1 - (p_val_two / 2)
    print("⚠️ [안내] SciPy 버전이 낮아 p-value를 수동 계산했습니다.")

# 3. 결과 출력
diff_mean = (before - after).mean()

print("-" * 40)
print(f"[분석 결과]")
print(f"1. 교육 전 평균: {before.mean():.4f}분")
print(f"2. 교육 후 평균: {after.mean():.4f}분")
print(f"3. 평균 차이(Before - After): {diff_mean:.4f}분")
print(f"4. t-통계량: {t_stat:.4f}")
print(f"5. p-value: {p_val:.15f}") # 매우 작은 값 확인을 위해 소수점 확장
print("-" * 40)

# 4. 최종 판정
if p_val < 0.05:
    print("결론: p-value < 0.05 이므로 귀무가설을 기각합니다.")
    print("=> 교육 후 보고서 작성 시간이 통계적으로 유의미하게 감소하였습니다.")
else:
    print("결론: p-value >= 0.05 이므로 귀무가설을 채택합니다.")
    print("=> 교육 전후 업무 시간 차이가 통계적으로 유의미하지 않습니다.")

⚠️ [안내] CSV 파일이 없어 테스트용 데이터를 생성하여 분석을 진행합니다.

----------------------------------------
[분석 결과]
1. 교육 전 평균: 61.7625분
2. 교육 후 평균: 56.6373분
3. 평균 차이(Before - After): 5.1251분
4. t-통계량: 13.8421
5. p-value: 0.000000000000308
----------------------------------------
결론: p-value < 0.05 이므로 귀무가설을 기각합니다.
=> 교육 후 보고서 작성 시간이 통계적으로 유의미하게 감소하였습니다.
